# Semi out-of-core

In [ ]:
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import metrics
from sklearn.datasets import fetch_20newsgroups

# Load dataset
newsgroups_train = fetch_20newsgroups(subset='train')
newsgroups_test = fetch_20newsgroups(subset='test')

# Initialize CountVectorizer
vectorizer = CountVectorizer()

# Initialize the Multinomial Naive Bayes classifier
clf = MultinomialNB()

# Batch size
batch_size = 100

# Fit the CountVectorizer to the entire vocabulary
vectorizer.fit(newsgroups_train.data)

# Train the model using small batches of data (out-of-core learning)
for i in range(0, len(newsgroups_train.data), batch_size):
    texts = newsgroups_train.data[i:i + batch_size]
    targets = newsgroups_train.target[i:i + batch_size]

    X_train = vectorizer.transform(texts)

    if i == 0:
        # For the first batch, we must provide all possible classes
        clf.partial_fit(
            X_train,
            targets,
            classes=np.unique(newsgroups_train.target)
        )
    else:
        clf.partial_fit(X_train, targets)

# Evaluate the model on the test data
X_test = vectorizer.transform(newsgroups_test.data)
predicted = clf.predict(X_test)

print("Accuracy:", metrics.accuracy_score(newsgroups_test.target, predicted))


### Important note (conceptual clarity)

- This code uses partial_fit, but because of this line:
- vectorizer.fit(newsgroups_train.data)
- The vocabulary is still built in-memory, so this is semi out-of-core, not fully out-of-core.

----

# TRUE Out-of-Core Multinomial Naïve Bayes

In [ ]:
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn import metrics
from sklearn.datasets import fetch_20newsgroups

# Load dataset
newsgroups_train = fetch_20newsgroups(subset='train')
newsgroups_test = fetch_20newsgroups(subset='test')

# TRUE out-of-core vectorizer (no fit, no vocab)
vectorizer = HashingVectorizer(
    n_features=2**20,
    alternate_sign=False
)

# Initialize classifier
clf = MultinomialNB()

# Batch size
batch_size = 100

classes = np.unique(newsgroups_train.target)

# Incremental training
for i in range(0, len(newsgroups_train.data), batch_size):
    texts = newsgroups_train.data[i:i + batch_size]
    targets = newsgroups_train.target[i:i + batch_size]

    X_train = vectorizer.transform(texts)

    if i == 0:
        clf.partial_fit(X_train, targets, classes=classes)
    else:
        clf.partial_fit(X_train, targets)

# Evaluation
X_test = vectorizer.transform(newsgroups_test.data)
predicted = clf.predict(X_test)

print("Accuracy:", metrics.accuracy_score(newsgroups_test.target, predicted))
